# EX_16: RPS Classification — 모델 크기 극소화
## MobileNetV3-Small + Pruning + QAT

| 항목 | EX_14 (MobileNetV2) | EX_16 (MobileNetV3-Small) |
|------|---------------------|---------------------------|
| 목표 | 크기/정확도 균형 | **크기 극소화 (1MB 이하 목표)** |
| 백본 | MobileNetV2 (3.4M) | **MobileNetV3-Small (2.5M)** |
| 경량화 기법 | QAT만 | **Pruning(레이어 선택) → QAT** |
| Pruning 방식 | 없음 | **Dense/Conv2D만 선택적 적용** |
| 목표 크기 | 5.12 MB | **1 MB 이하** |

### 경량화 파이프라인
```
Phase 1: Head 학습 (백본 고정)
Phase 2: 전체 Fine-tuning
Phase 3: Pruning (Dense/Conv2D 선택적 적용, 70% 제거)
Phase 4: QAT (INT8 양자화)
→ TFLite 변환
```

## Step 0. 패키지 설치
> ⚠️ 설치 후 **런타임 재시작** 필수

In [ ]:
!pip install tensorflow-model-optimization -q

## Step 1. 모듈 로딩

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import glob, cv2, pathlib, shutil, math
from sklearn.model_selection import train_test_split
import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print('NumPy      :', np.__version__)
print('TensorFlow :', tf.__version__)
print('GPU        :', tf.config.list_physical_devices('GPU'))

## Step 2. Google Drive 마운트

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')
    colab = True
except:
    print('Local environment.')
    colab = False

## Step 3. 데이터셋 준비

In [ ]:
files_path  = '/content/drive/MyDrive/sds/RPS_Dataset'
IMG_SIZE    = 224
NUM_CLASSES = 3
class_names = ['Rock', 'Paper', 'Scissors']

print(f'데이터 로딩 중... (IMG_SIZE={IMG_SIZE})')
all_images, all_labels = [], []

for label in range(NUM_CLASSES):
    files = glob.glob(f'{files_path}/{label}/*.*')
    print(f'  [{label}] {class_names[label]}: {len(files)}개')
    for f in files:
        img = cv2.imread(f, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        all_images.append(img)
        all_labels.append(label)

all_images = np.array(all_images, dtype=np.uint8)
all_labels = np.array(all_labels)

train_data, test_data, train_labels, test_labels = train_test_split(
    all_images, all_labels,
    test_size=0.2, random_state=123, stratify=all_labels
)

print(f'\nTrain: {train_data.shape}, Test: {test_data.shape}')
print(f'클래스 분포 (train): {np.bincount(train_labels)}')
print(f'클래스 분포 (test) : {np.bincount(test_labels)}')

def mobilenet_preprocess(x):
    return (x.astype(np.float32) / 127.5) - 1.0

train_data_f  = train_data.astype(np.float32)
test_data_pre = mobilenet_preprocess(test_data)
print('데이터 준비 완료')

## Step 4. 데이터 증강

In [ ]:
def preprocess_and_augment(x):
    return mobilenet_preprocess(x)

img_gen_train = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_and_augment,
    horizontal_flip=True,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    brightness_range=(0.7, 1.3),
    zoom_range=0.2,
    shear_range=10,
    fill_mode='nearest'
)

def make_gen():
    return img_gen_train.flow(train_data_f, train_labels, batch_size=32, seed=1)

print('ImageDataGenerator 준비 완료')

## Step 5. MobileNetV3-Small 모델 구성 (Phase 1: 백본 고정)

In [ ]:
inputs     = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
base_model = keras.applications.MobileNetV3Small(
    weights='imagenet',
    include_top=False,
    input_tensor=inputs,
    include_preprocessing=False
)
base_model.trainable = False

x = base_model.output
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs)

trainable_cnt = sum([np.prod(v.shape) for v in model.trainable_variables])
print(f'[Phase 1] 학습 가능: {trainable_cnt:,} / 전체: {model.count_params():,}')

## Step 6. Phase 1: Head 학습 (백본 고정)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

print('=== Phase 1: Head 학습 (백본 고정) ===')
history_p1 = model.fit(
    make_gen(),
    epochs=10,
    validation_data=(test_data_pre, test_labels),
    callbacks=[
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy', factor=0.5, patience=3, verbose=1, min_lr=1e-6
        )
    ]
)

_, acc_p1 = model.evaluate(test_data_pre, test_labels, verbose=0)
print(f'\nPhase 1 완료 - 정확도: {acc_p1:.4f}')

## Step 7. Phase 2: 전체 Fine-tuning

In [ ]:
for layer in base_model.layers:
    if not isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = True

trainable_cnt = sum([np.prod(v.shape) for v in model.trainable_variables])
print(f'[Phase 2] 학습 가능: {trainable_cnt:,}')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

savedModelName_p2 = 'EX16_RPS_MobileNetV3Small_best.h5'

print('\n=== Phase 2: 전체 Fine-tuning ===')
history_p2 = model.fit(
    make_gen(),
    epochs=30,
    validation_data=(test_data_pre, test_labels),
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            savedModelName_p2, save_best_only=True, monitor='val_accuracy', verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy', factor=0.5, patience=4, verbose=1, min_lr=1e-8
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1
        )
    ]
)

_, acc_p2 = model.evaluate(test_data_pre, test_labels, verbose=0)
print(f'\nPhase 2 완료 - 정확도: {acc_p2:.4f}')

## Step 8. Phase 3: Pruning (레이어 선택적 적용)
- `TFOpLambda` 등 지원 불가 레이어가 있어 모델 전체 적용 불가
- `Dense` / `Conv2D` / `DepthwiseConv2D` 레이어만 선택적으로 Pruning
- `clone_model`로 재구성 → Pruning 적용 가능한 구조로 변환
- `final_sparsity=0.70` : 가중치 70% 제거

In [ ]:
batch_size      = 32
epochs_pruning  = 10
steps_per_epoch = math.ceil(len(train_data) / batch_size)
end_step        = steps_per_epoch * epochs_pruning

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.30,
    final_sparsity=0.70,
    begin_step=0,
    end_step=end_step
)

# 지원 가능한 레이어만 선택적으로 Pruning 적용
# TFOpLambda / Reshape 등 지원 불가 레이어는 그대로 통과
PRUNABLE = (
    keras.layers.Dense,
    keras.layers.Conv2D,
    keras.layers.DepthwiseConv2D
)

def apply_pruning_to_layer(layer):
    if isinstance(layer, PRUNABLE):
        return tfmot.sparsity.keras.prune_low_magnitude(
            layer, pruning_schedule=pruning_schedule
        )
    return layer

model_for_pruning = keras.models.clone_model(
    model,
    clone_function=apply_pruning_to_layer
)

# clone 후 가중치 복사 (학습된 가중치 유지)
model_for_pruning.set_weights(model.get_weights())

model_for_pruning.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# Pruning 대상 레이어 수 확인
pruned_cnt = sum(
    1 for layer in model_for_pruning.layers
    if isinstance(layer, tfmot.sparsity.keras.PruneLowMagnitude)
)
print(f'Pruning 적용 레이어 수: {pruned_cnt}')

print('\n=== Phase 3: Pruning (70% 가중치 제거) ===')
history_pruning = model_for_pruning.fit(
    make_gen(),
    epochs=epochs_pruning,
    validation_data=(test_data_pre, test_labels),
    callbacks=[
        tfmot.sparsity.keras.UpdatePruningStep(),   # 매 스텝 sparsity 업데이트 (필수)
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy', factor=0.5, patience=3, verbose=1, min_lr=1e-8
        )
    ]
)

_, acc_pruning = model_for_pruning.evaluate(test_data_pre, test_labels, verbose=0)
print(f'\nPruning 완료 - 정확도: {acc_pruning:.4f}')

# Pruning wrapper 제거 → 실제 sparse 가중치 모델
model_pruned = tfmot.sparsity.keras.strip_pruning(model_for_pruning)
print('Pruning wrapper 제거 완료')

# 실제 sparsity 확인
total_w, zero_w = 0, 0
for layer in model_pruned.layers:
    for w in layer.weights:
        arr = w.numpy()
        total_w += arr.size
        zero_w  += np.sum(arr == 0)
print(f'실제 Sparsity: {zero_w/total_w*100:.1f}% ({zero_w:,}/{total_w:,})')

## Step 9. Phase 4: QAT (양자화 인식 학습)
- Pruned 모델에 QAT 적용
- `Dense + Conv2D + DepthwiseConv2D` 선택적 양자화
- `lr=1e-6`, 5 epochs

In [ ]:
def apply_quantization_to_layer(layer):
    if isinstance(layer, PRUNABLE):
        return tfmot.quantization.keras.quantize_annotate_layer(layer)
    return layer

annotated_model = keras.models.clone_model(
    model_pruned,
    clone_function=apply_quantization_to_layer
)
qat_model = tfmot.quantization.keras.quantize_apply(annotated_model)

qat_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-6),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

savedModelName_qat = 'EX16_RPS_MobileNetV3Small_QAT.h5'

print('=== Phase 4: QAT Fine-tuning ===')
history_qat = qat_model.fit(
    make_gen(),
    epochs=5,
    validation_data=(test_data_pre, test_labels),
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            savedModelName_qat, save_best_only=True, monitor='val_accuracy', verbose=1
        )
    ]
)

_, acc_qat = qat_model.evaluate(test_data_pre, test_labels, verbose=0)
print(f'\nQAT 완료 - 정확도: {acc_qat:.4f}')

## Step 10. 전체 학습 곡선 시각화

In [ ]:
acc_all      = history_p1.history['accuracy']     + history_p2.history['accuracy']     + history_pruning.history['accuracy']     + history_qat.history['accuracy']
val_acc_all  = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy'] + history_pruning.history['val_accuracy'] + history_qat.history['val_accuracy']
loss_all     = history_p1.history['loss']         + history_p2.history['loss']         + history_pruning.history['loss']         + history_qat.history['loss']
val_loss_all = history_p1.history['val_loss']     + history_p2.history['val_loss']     + history_pruning.history['val_loss']     + history_qat.history['val_loss']

p1_end    = len(history_p1.history['accuracy'])
p2_end    = p1_end + len(history_p2.history['accuracy'])
prune_end = p2_end + len(history_pruning.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('EX_16 학습 곡선 (Phase1 → Phase2 → Pruning → QAT)', fontsize=14)

vlines = [('r', p1_end, 'P1→P2'), ('orange', p2_end, 'P2→Pruning'), ('purple', prune_end, 'Pruning→QAT')]

for ax, (y_train, y_val, title) in zip(axes, [
    (acc_all, val_acc_all, 'Accuracy'),
    (loss_all, val_loss_all, 'Loss')
]):
    ax.plot(y_train, 'b', label='Train')
    ax.plot(y_val,   'g', label='Validation')
    for color, x_pos, lbl in vlines:
        ax.axvline(x=x_pos - 0.5, color=color, linestyle='--', label=lbl)
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True)

plt.tight_layout()
plt.show()

## Step 11. 최적 QAT 모델 로딩 및 예측 시각화

In [ ]:
from tensorflow_model_optimization.quantization.keras import quantize_scope

with quantize_scope():
    model_best = keras.models.load_model(savedModelName_qat)

y_out  = model_best.predict(test_data_pre)
y_pred = np.argmax(y_out, axis=1)

size = test_data.shape[0] // 10
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('EX_16: MobileNetV3-Small + Pruning + QAT 예측 결과 (녹색=정답, 적색=오답)', fontsize=12)

for idx in range(10):
    cnt = idx * size + size // 2
    ax  = axes[idx // 5, idx % 5]
    ax.imshow(test_data[cnt])
    color = 'green' if y_pred[cnt] == test_labels[cnt] else 'red'
    ax.set_title(
        f'GT: {class_names[test_labels[cnt]]}\nPred: {class_names[y_pred[cnt]]}',
        color=color, fontsize=10
    )
    ax.axis('off')

plt.tight_layout()
plt.show()

overall_acc = (y_pred == test_labels).mean()
print(f'\n전체 정확도: {overall_acc:.4f}')

## Step 12. TFLite 변환 (INT8)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_best)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

print(f'TFLite 모델 크기: {len(tflite_model):,} bytes ({len(tflite_model)/1024/1024:.2f} MB)')

save_dir = '/content/drive/MyDrive/files/save/' if colab else '../files/save/'
pathlib.Path(save_dir).mkdir(exist_ok=True, parents=True)

tfliteFileName = 'EX16_RPS_MobileNetV3Small_Pruning_QAT.tflite'
with open(save_dir + tfliteFileName, 'wb') as f:
    f.write(tflite_model)

shutil.copy(savedModelName_qat, save_dir + savedModelName_qat)
print(f'저장 완료: {save_dir}')

## Step 13. TFLite 정확도 평가

In [ ]:
def evaluate_tflite(model_content, test_data, test_labels):
    interpreter = tf.lite.Interpreter(model_content=model_content)
    interpreter.allocate_tensors()
    input_details  = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    correct = 0
    for image, label in zip(test_data, test_labels):
        img = np.expand_dims(image.astype(np.float32), axis=0)
        interpreter.set_tensor(input_details['index'], img)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])
        if np.argmax(output) == label:
            correct += 1
    return correct / len(test_labels)

ex16_tflite_acc = evaluate_tflite(tflite_model, test_data_pre, test_labels)
print(f'EX_16 TFLite 정확도: {ex16_tflite_acc:.4f}')

## Step 14. 최종 비교

In [ ]:
results = [
    ('EX_12: DenseNet121 + QAT',                     0.9927, 7_703_840),
    ('EX_14: MobileNetV2 + QAT (224px)',             0.8860, int(5.12*1024*1024)),
    ('EX_15: EfficientNetB3 (정확도 극대화)',          0.0,    0),   # 실험 결과로 교체
    ('EX_16: MobileNetV3S + Pruning + QAT (경량화)', ex16_tflite_acc, len(tflite_model)),
]

print('=' * 68)
print(f'{"모델":<46} {"정확도":>8} {"크기(MB)":>10}')
print('-' * 68)
for name, acc, sz in results:
    size_str = f'{sz/1024/1024:.2f}' if sz > 0 else '  N/A'
    print(f'{name:<46} {acc:>8.4f} {size_str:>10}')
print('=' * 68)

print('\n[EX_16 경량화 파이프라인]')
print('  MobileNetV3-Small → Fine-tuning → Pruning 70% → QAT INT8 → TFLite')
print(f'  EX_12 대비 크기: {len(tflite_model)/7_703_840*100:.0f}%')